Загружаем датасет

In [1]:
from sqlalchemy import create_engine
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv

def create_connection():

    load_dotenv()
    host = os.environ.get('DB_DESTINATION_HOST')
    port = os.environ.get('DB_DESTINATION_PORT')
    db = os.environ.get('DB_DESTINATION_NAME')
    username = os.environ.get('DB_DESTINATION_USER')
    password = os.environ.get('DB_DESTINATION_PASSWORD')

    conn = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{db}')
    return conn

conn = create_connection()
data = pd.read_sql("select * from flats_price", conn)
data

/tmp/ipykernel_2925/1390673944.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


,id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.90,19.900000,1,False,False,35.099998,9500000,1965,6,55.717113,37.781120,2.64,84,12,True
1,1,18012,7,0.00,16.600000,1,False,False,43.000000,13500000,2001,2,55.794849,37.608013,3.00,97,10,True
2,2,17821,9,9.00,32.000000,2,False,False,56.000000,13500000,2000,4,55.740040,37.761742,2.70,80,10,True
3,3,18579,1,10.10,43.099998,3,False,False,76.000000,20000000,2002,4,55.672016,37.570877,2.64,771,17,True
4,4,9293,3,3.00,14.000000,1,False,False,24.000000,5200000,1971,1,55.808807,37.707306,2.60,208,9,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141357,141357,22455,16,11.00,18.000000,1,False,False,42.000000,10500000,2013,4,55.626579,37.313503,2.64,672,25,True
141358,141358,3162,5,5.28,28.330000,2,False,False,41.110001,7400000,1960,1,55.727470,37.768677,2.48,80,5,False
141359,141359,6513,7,5.30,20.000000,1,False,False,31.500000,9700000,1966,4,55.704315,37.506584,2.64,72,9,True
141360,141360,23952,15,13.80,33.700001,2,False,False,65.300003,11750000,2017,4,55.699863,37.939564,2.70,480,25,True


Дубликаты

Дубликаты по id

In [2]:
is_duplicated_id = data.duplicated(subset=['id'], keep=False)
print(sum(is_duplicated_id)) 

0


In [3]:
feature_cols = data.drop(['id'], axis=1).columns.tolist()
is_duplicated_features = data.duplicated(subset = feature_cols, keep=False)
print(sum(is_duplicated_features))

17425


0 дубликатов по id и 17425 полных дубликатов по признакам. Их доля значительна, поэтому оставим оригинал(оригинал будем считать первую попавшуюся).

In [4]:
def remove_duplicates(data):
    feature_cols = data.columns.drop('id').tolist()
    is_duplicated_features = data.duplicated(subset=feature_cols, keep='first')
    data = data[~is_duplicated_features].reset_index(drop=True)
    return data 
remove_duplicates(data)

,id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.90,19.900000,1,False,False,35.099998,9500000,1965,6,55.717113,37.781120,2.64,84,12,True
1,1,18012,7,0.00,16.600000,1,False,False,43.000000,13500000,2001,2,55.794849,37.608013,3.00,97,10,True
2,2,17821,9,9.00,32.000000,2,False,False,56.000000,13500000,2000,4,55.740040,37.761742,2.70,80,10,True
3,3,18579,1,10.10,43.099998,3,False,False,76.000000,20000000,2002,4,55.672016,37.570877,2.64,771,17,True
4,4,9293,3,3.00,14.000000,1,False,False,24.000000,5200000,1971,1,55.808807,37.707306,2.60,208,9,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132241,141357,22455,16,11.00,18.000000,1,False,False,42.000000,10500000,2013,4,55.626579,37.313503,2.64,672,25,True
132242,141358,3162,5,5.28,28.330000,2,False,False,41.110001,7400000,1960,1,55.727470,37.768677,2.48,80,5,False
132243,141359,6513,7,5.30,20.000000,1,False,False,31.500000,9700000,1966,4,55.704315,37.506584,2.64,72,9,True
132244,141360,23952,15,13.80,33.700001,2,False,False,65.300003,11750000,2017,4,55.699863,37.939564,2.70,480,25,True


### Пропуски

In [5]:
data.isnull().sum()

id                   0
building_id          0
floor                0
kitchen_area         0
living_area          0
rooms                0
is_apartment         0
studio               0
total_area           0
price                0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
flats_count          0
floors_total         0
has_elevator         0
dtype: int64

Пропусков нет, но могут появится в будущем, выгоднее будет их тогда удалить, так как их число будет мало по сравнению с выборокой.

In [6]:
def remove_missing_values(data: pd.DataFrame) -> pd.DataFrame:
    return data.dropna().reset_index(drop=True)
remove_missing_values(data)


,id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.90,19.900000,1,False,False,35.099998,9500000,1965,6,55.717113,37.781120,2.64,84,12,True
1,1,18012,7,0.00,16.600000,1,False,False,43.000000,13500000,2001,2,55.794849,37.608013,3.00,97,10,True
2,2,17821,9,9.00,32.000000,2,False,False,56.000000,13500000,2000,4,55.740040,37.761742,2.70,80,10,True
3,3,18579,1,10.10,43.099998,3,False,False,76.000000,20000000,2002,4,55.672016,37.570877,2.64,771,17,True
4,4,9293,3,3.00,14.000000,1,False,False,24.000000,5200000,1971,1,55.808807,37.707306,2.60,208,9,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141357,141357,22455,16,11.00,18.000000,1,False,False,42.000000,10500000,2013,4,55.626579,37.313503,2.64,672,25,True
141358,141358,3162,5,5.28,28.330000,2,False,False,41.110001,7400000,1960,1,55.727470,37.768677,2.48,80,5,False
141359,141359,6513,7,5.30,20.000000,1,False,False,31.500000,9700000,1966,4,55.704315,37.506584,2.64,72,9,True
141360,141360,23952,15,13.80,33.700001,2,False,False,65.300003,11750000,2017,4,55.699863,37.939564,2.70,480,25,True


### Выбросы

Посмотрим сначала на колонки  типа int

In [7]:
num_features = data.drop(columns=['id', 'building_id']).select_dtypes(include=['int'])
num_features.columns

Index(['floor', 'rooms', 'price', 'build_year', 'building_type_int',
       'flats_count', 'floors_total'],
      dtype='object')

Из них интерес представляет только столбец цен из-за широкого диапозано значений.

In [8]:
import seaborn as sns

display(num_features['price'].describe())

count    1.413620e+05
mean     1.944162e+07
std      6.626954e+07
min      1.100000e+01
25%      8.900000e+06
50%      1.185000e+07
75%      1.695000e+07
max      9.873738e+09
Name: price, dtype: float64

Максимальное значение существенно превышает среднее, что свидетельствует о наличии объектов с очень высокой стоимостью. Поэтому столбец `price` будем рассматривать при анализе выбросов вместе с непрерывными признаками.


In [9]:
num_cols = data.select_dtypes(['float']).columns.tolist()
num_cols.append("price")
threshold = 1.5
potential_outliers = pd.DataFrame()

for col in num_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    margin = IQR * threshold
    lower = Q1 - margin
    upper = Q3 + margin
    potential_outliers[col] = ~data[col].between(lower, upper)

outliers = potential_outliers.any(axis=1)

print(potential_outliers.sum())
print(outliers.sum())
print(outliers.mean())

kitchen_area       8076
living_area        4962
total_area         7839
latitude            111
longitude          1975
ceiling_height     9989
price             14318
dtype: int64
24073
0.17029328956862524


Выбросов оказалось много 17% от выборки, если всех их удаить, потеряем немалую часть данных. Поменяем значение threshold

In [10]:
for threshold in [1.5, 2, 2.5, 3]:
    potential_outliers = pd.DataFrame()
    for col in num_cols:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        margin = IQR * threshold
        lower = Q1 - margin
        upper = Q3 + margin
        potential_outliers[col] = ~data[col].between(lower, upper)

    outliers = potential_outliers.any(axis=1)
    print(threshold)
    print(outliers.sum())
    print(outliers.mean())

1.5
24073
0.17029328956862524
2
18193
0.12869795277372986
2.5
15330
0.10844498521526294
3
11430
0.08085624142273029


При threshold = 3, выбросами было призанано примерно 8%, поэтому данный порог был выбран как менее агрессивный.

In [11]:
def remove_outliers(data: pd.DataFrame) -> pd.DataFrame:
    potential_outliers = pd.DataFrame()
    num_cols = data.select_dtypes(['float']).columns.tolist()
    num_cols.append("price")
    threshold = 3
    for col in num_cols:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        margin = IQR * threshold
        lower = Q1 - margin
        upper = Q3 + margin
        potential_outliers[col] = ~data[col].between(lower, upper)
    outliers = potential_outliers.any(axis=1)
    data = data[~outliers]
    return data
data = remove_duplicates(data)
data = remove_missing_values(data)
data = remove_outliers(data)
data.shape

(121766, 18)